# Ambiente de Testes SQL com DuckDB

Este notebook é uma plataforma de laboratório para testar consultas SQL com DuckDB de forma ampla e profissional.

- Carrega automaticamente todos os arquivos CSV na pasta `data/`.
- Registra cada arquivo como uma tabela DuckDB para criar um ambiente relacional completo.
- Permite criar e validar consultas de seleção, agregação, joins, funções de janela e análises exploratórias.

> Além disso, existe uma pasta `sql_commands/` no repositório para que você salve seus scripts e exemplos SQL em arquivos `.sql` ou `.txt`. Isso torna seu material de aula mais organizado e reutilizável.


## 1. Preparar o ambiente

Execute esta célula para carregar as bibliotecas necessárias e criar uma função de apoio para executar consultas SQL no DuckDB.


In [8]:
import importlib.util
import subprocess
import sys

for pkg in ('duckdb', 'pandas'):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

import duckdb
import pandas as pd
from pathlib import Path

# Diretório de dados local
data_dir = Path('data')
if not data_dir.exists():
    raise FileNotFoundError(f'Diretório não encontrado: {data_dir}')

csv_files = sorted(data_dir.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(f'Nenhum arquivo CSV encontrado em: {data_dir}')

# Conexão local em memória
con = duckdb.connect(database=':memory:')
loaded_tables = []
for csv_path in csv_files:
    table_name = csv_path.stem.lower()
    df = pd.read_csv(csv_path)
    con.register(table_name, df)
    loaded_tables.append(table_name)

last_result = None

def sql(query: str):
    global last_result
    last_result = con.execute(query).df()
    display(last_result)

## 2. Carregar os dados locais

Este notebook carrega todos os arquivos CSV presentes em `data/`.
Cada arquivo é registrado automaticamente no DuckDB como uma tabela cujo nome é o nome do arquivo (sem extensão).

Dessa forma, você pode testar `JOINs` e outras consultas SQL entre tabelas diferentes.

Por exemplo:

- `cliente.csv` → tabela `cliente`
- `produto.csv` → tabela `produto`
- `vendedor.csv` → tabela `vendedor`
- `vendas.csv` → tabela `vendas`

In [9]:
print(f'Arquivos CSV carregados como tabelas: {", ".join(loaded_tables)}')
print(f'Tabela de exemplo: {loaded_tables[0]}')
display(con.execute(f"SELECT * FROM {loaded_tables[0]} LIMIT 5").df())


Arquivos CSV carregados como tabelas: baby_names
Tabela de exemplo: baby_names


,Sexo,Nome,Total
0,Menina,Ava,95
1,Menina,Emma,106
2,Menino,Ethan,115
3,Menina,Isabella,100
4,Menino,Jacob,101


## 3. Playground SQL

Use esta célula para testar consultas básicas e explorar os dados.
Altere a query conforme desejar e rode novamente para ver os resultados.

In [10]:
sql(
    '''
    SELECT Nome, Sexo, Total
    FROM baby_names
    ORDER BY Total DESC
    LIMIT 10
    '''
)


,Nome,Sexo,Total
0,Noah,Menino,120
1,Ethan,Menino,115
2,Emma,Menina,106
3,Jacob,Menino,101
4,Olivia,Menina,100
5,Isabella,Menina,100
6,Ava,Menina,95
7,Sophia,Menina,88
8,Liam,Menino,84
9,Logan,Menino,73


## 4. Consultas de agregação

Exemplos de análise de grupos, contagem e soma por sexo.

In [11]:
sql(
    '''
    SELECT Sexo, COUNT(*) AS total_nomes, SUM(Total) AS total_bebes
    FROM baby_names
    GROUP BY Sexo
    ORDER BY total_bebes DESC
    '''
)


,Sexo,total_nomes,total_bebes
0,Menino,5,493.0
1,Menina,5,489.0


## 5. Funções de janela e ranking

A seguir vemos exemplos de `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()` e `NTILE()`.
Essas funções ajudam a ranquear registros dentro de partições e a criar ordenações avançadas.

In [12]:
sql(
    '''
    SELECT
      Nome,
      Sexo,
      Total,
      ROW_NUMBER() OVER (PARTITION BY Sexo ORDER BY Total DESC) AS row_number_por_sexo,
      RANK() OVER (PARTITION BY Sexo ORDER BY Total DESC) AS rank_por_sexo,
      DENSE_RANK() OVER (PARTITION BY Sexo ORDER BY Total DESC) AS dense_rank_por_sexo,
      NTILE(4) OVER (PARTITION BY Sexo ORDER BY Total DESC) AS quartil_por_sexo
    FROM baby_names
    ORDER BY Sexo, Total DESC
    LIMIT 20
    '''
)


,Nome,Sexo,Total,row_number_por_sexo,rank_por_sexo,dense_rank_por_sexo,quartil_por_sexo
0,Emma,Menina,106,1,1,1,1
1,Isabella,Menina,100,2,2,2,1
2,Olivia,Menina,100,3,2,2,2
3,Ava,Menina,95,4,4,3,3
4,Sophia,Menina,88,5,5,4,4
5,Noah,Menino,120,1,1,1,1
6,Ethan,Menino,115,2,2,2,1
7,Jacob,Menino,101,3,3,3,2
8,Liam,Menino,84,4,4,4,3
9,Logan,Menino,73,5,5,5,4


## 6. Exercícios sugeridos

1. Encontre os 5 nomes mais populares por ano.
2. Use `RANK()` para comparar nomes que empatam no mesmo Total.
3. Crie um relatório com `NTILE(3)` mostrando três faixas de popularidade por sexo.